# To Be or Not To Be

## Imports and Byte-Pair Encoding
- Using tinyshakespeare dataset
- Strip line breaks
- Lowercase everything

In [1]:
!uv pip install torch tokenizers requests ipykernel

Audited 4 packages in 33ms


In [2]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
text = text.lower()

In [3]:
print(text[:300])

first citizen:
before we proceed any further, hear me speak.

all:
speak, speak.

first citizen:
you are all resolved rather to die than to famish?

all:
resolved. resolved.

first citizen:
first, you know caius marcius is chief enemy to the people.

all:
we know't, we know't.

first citizen:
let us


In [4]:
tokens = list(text)
vocabulary = list(set(tokens))
print("number of tokens in corpus", len(tokens))
print("vocabulary size", len(vocabulary))
print(vocabulary)

number of tokens in corpus 1115394
vocabulary size 39
['u', '!', 'x', '.', '-', 't', 'r', 'f', '3', 'y', 'o', 'm', 'e', 'w', 'b', 'k', 'a', 'v', 'i', '\n', ':', '?', ',', 'c', '$', 'g', 's', 'd', '&', 'q', 'h', ';', "'", 'j', 'n', 'z', 'l', ' ', 'p']


In [5]:
vocab_size = 1_000
def byte_pair_encode(tokens, vocabulary, vocab_size=1_000):
    """
    Isaac: I copied and pasted this implementation of BPE from ChatGPT.
    It uses recursion with a base case of returning the vocabulary once it's matched the vocab size.
    For the "recursive step" (idk proper, kinda like inductivive step in math),
    it adds the most common pair of tokens to the vocab and merges them in the corpus.
    I had a buggy implementation of the same idea that I had GPT 5.6 Sol (Medium) correct.

    It was annoyingly slow though, so I just ended up using BPE from HuggingFace, which is in Rust. :D
    """
    # Base case
    if len(vocabulary) >= vocab_size:
        return tokens, vocabulary

    # Count adjacent pairs
    count_by_token_pair = {}

    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        count_by_token_pair[pair] = count_by_token_pair.get(pair, 0) + 1

    # Find most common pair
    most_used_pair = max(
        count_by_token_pair,
        key=count_by_token_pair.get
    )

    # Add new token
    new_token = most_used_pair[0] + most_used_pair[1]
    vocabulary.append(new_token)

    # Merge occurrences
    new_tokens = []
    i = 0

    while i < len(tokens):
        if (
            i < len(tokens) - 1
            and (tokens[i], tokens[i + 1]) == most_used_pair
        ):
            new_tokens.append(new_token)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1

    # Return recursion
    return byte_pair_encode(new_tokens, vocabulary, vocab_size)

# tokens, vocabulary = byte_pair_encode(tokens, vocabulary, vocab_size)

In [6]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

In [7]:
"""
I'm using HF's BPE (implemented in Rust) because I don't want to wait
for so long every time I run this program from scratch.
"""

def byte_pair_encode_fast(tokens, vocabulary, vocab_size=1_000):
    text = "".join(tokens)

    tokenizer = Tokenizer(BPE())

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        initial_alphabet=vocabulary
    )

    tokenizer.train_from_iterator([text], trainer=trainer)

    # BPE-tokenized corpus
    tokens = tokenizer.encode(text).tokens

    # Vocabulary as a list ordered by token ID
    vocab_dict = tokenizer.get_vocab()
    vocabulary = [
        token
        for token, idx in sorted(vocab_dict.items(), key=lambda x: x[1])
    ]

    return tokens, vocabulary


tokens, vocabulary = byte_pair_encode_fast(
    tokens,
    vocabulary,
    vocab_size
)

In [10]:
"""
Isaac:
Saving the python objects to file before loading them back up
.pkl is the extention for python objects.
wb = write binary, rb = read binary
"""

import pickle

with open("bpe_data.pkl", "wb") as f:
    pickle.dump((tokens, vocabulary), f)

In [11]:
with open("bpe_data.pkl", "rb") as f:
    tokens, vocabulary = pickle.load(f)

## Create Dataset
Consider this the function comment for the cell below from Isaac
- Training pairs (token[i], token[i+k]) -> k
- For i going from 0 to len(token)-k
- For k going from 0 to 10
- We predict word distance since it might actually matter
- I'm using Pytorch since I like its style and have some experience with it

In [ ]:
import torch
from torch.utils.data import TensorDataset

def create_dataset(tokens, token_to_idx, max_distance=10):
    X = []
    y = []

    for k in range(1, max_distance + 1):
        for i in range(len(tokens) - k):
            X.append([
                token_to_idx[tokens[i]],
                token_to_idx[tokens[i + k]]
            ])
            y.append(k)

    X = torch.tensor(X, dtype=torch.long)
    y = torch.tensor(y, dtype=torch.long)

    return TensorDataset(X, y)

In [ ]:
# Isaac: Making a dictionary to get the indice of each token (for future one-hot encoding)
token_to_idx = {token: i for i, token in enumerate(vocabulary)}

"""
Creating a train-validation-test split
"""

from torch.utils.data import random_split, DataLoader

dataset = create_dataset(tokens, token_to_idx)

# GPT wrote this comment, but I set the values: make a split of 70% train, 10% validation, 20% test
train_size = int(0.7 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

torch.manual_seed(0)
train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
)

# Using data loader to convert to nice format for loading, training, and evaluating (with batches for arrellization)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## Train the Embedder

Consider this the function comment for the cell below from Isaac
- Embed token[i] and embed token[i+k]: shape vocab_size by 128 (embedding dimension)
- Compute the dot product of the embeded tokens (ideally it's k).
- Do gradient descent from the ideal value of k (based on MSE loss)
- Notice how we actually just need to train the embedding

In [ ]:
import torch.nn as nn

In [ ]:
embedding_dim = 128

# Shape conceptually: (vocab_size, embedding_dim)
embedding = nn.Embedding(vocab_size, embedding_dim)

optimizer = torch.optim.Adam(embedding.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(12):
    # Training
    embedding.train()

    for batch_idx, (X_batch, k_batch) in enumerate(train_loader):
        if batch_idx >= 16_384:
            break  # Isaac: It was taking too long so I decided to cut the training data short

        # Embed token[i] and token[i+k]
        e1 = embedding(X_batch[:, 0])  # (batch_size, embedding_dim)
        e2 = embedding(X_batch[:, 1])  # (batch_size, embedding_dim)

        # Dot product for each pair
        predicted_k = (e1 * e2).sum(dim=1)  # (batch_size,)

        # Ideally predicted_k == k
        loss = loss_fn(predicted_k, k_batch.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Validation
    embedding.eval()

    total_val_loss = 0
    total_val_examples = 0

    with torch.no_grad():
        for X_batch, k_batch in val_loader:
            e1 = embedding(X_batch[:, 0])
            e2 = embedding(X_batch[:, 1])

            predicted_k = (e1 * e2).sum(dim=1)

            val_loss = loss_fn(predicted_k, k_batch.float())

            batch_size = X_batch.size(0)
            total_val_loss += val_loss.item() * batch_size
            total_val_examples += batch_size

    average_val_loss = total_val_loss / total_val_examples

    print(
        f"Epoch {epoch}: "
        f"train loss = {loss.item():.4f}, "
        f"validation loss = {average_val_loss:.4f}"
    )

    """
    Isaac, based on ChatGPT's notes:
    train loss is based on average MSE loss for last batch
    while val loss is for average over all val examples
    """

In [ ]:
"""
Isaac: Evaluate the embedding matrix on the test set to see if the MSE error is similar
"""

embedding.eval()

total_loss = 0
total_examples = 0

with torch.no_grad():
    for X_batch, k_batch in test_loader:
        e1 = embedding(X_batch[:, 0])
        e2 = embedding(X_batch[:, 1])

        predicted_k = (e1 * e2).sum(dim=1)

        loss = loss_fn(predicted_k, k_batch.float())

        batch_size = X_batch.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

test_loss = total_loss / total_examples

print(f"Test MSE: {test_loss:.4f}")

## Making a Next-token Predictor

## Citation of AI Use
I used this ChatGPT chat for syntax while I prompted with my ideas: https://chatgpt.com/share/6abaeddf-4c24-83ea-9777-f0f85be4d542

All of my comments have my name on them. Otherwise, ChatGPT wrote the small comment for clarity.